In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings("ignore") # to avoid deprecation warnings
from sklearn.model_selection import train_test_split,cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression,LinearRegression, Lasso, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)


In [2]:
data=pd.read_csv('Walmart_Store_sales.csv')

In [3]:
data.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB


In [5]:
data.describe(include='all')

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,19-10-2012,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


In [6]:
(data.isnull().sum()/data.shape[0]*100).sort_values(ascending=False)

Date            12.000000
Temperature     12.000000
Unemployment    10.000000
Weekly_Sales     9.333333
Fuel_Price       9.333333
Holiday_Flag     8.000000
CPI              8.000000
Store            0.000000
dtype: float64

In [7]:
data['Date'] = pd.to_datetime(data['Date'], format='%d-%m-%Y')


In [8]:

check_outliers_list=['CPI', 'Fuel_Price', 'Unemployment', 'Temperature']
data[check_outliers_list].apply(lambda x: (x > (abs(x.mean()) + 3 * x.std())).sum())


CPI             0
Fuel_Price      0
Unemployment    5
Temperature     0
dtype: int64

In [9]:
px.scatter_matrix(data,width=1000,height=1000).show()

In [10]:
data=data.loc[data['Unemployment']<13]
px.scatter_matrix(data,width=1000,height=1000).show()

In [11]:
data['year'] = data.loc[:,'Date'].dt.year
data['month'] = data.loc[:,'Date'].dt.month
data['day'] = data.loc[:,'Date'].dt.day
data['day_of_week']=data.loc[:,'Date'].dt.dayofweek

In [12]:
data=data.loc[~data['Weekly_Sales'].isnull()]
(data.isnull().sum()/data.shape[0]*100).sort_values(ascending=False)

Date            12.820513
year            12.820513
month           12.820513
day             12.820513
day_of_week     12.820513
Temperature      9.401709
Fuel_Price       9.401709
Holiday_Flag     8.547009
CPI              7.692308
Store            0.000000
Weekly_Sales     0.000000
Unemployment     0.000000
dtype: float64

# Preprocessing

In [13]:
# data=data.dropna(subset=['month'])

In [14]:
X=data.drop(columns=['Weekly_Sales','Date'])
y=data['Weekly_Sales']

In [15]:
categorical_features=['Store','Holiday_Flag']
numeric_features= [c for c in X.columns if c not in categorical_features]

print('categorical_features :',categorical_features)
print('numeric_features :',numeric_features)


categorical_features : ['Store', 'Holiday_Flag']
numeric_features : ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'year', 'month', 'day', 'day_of_week']


In [16]:
X.isnull().sum()/len(X)*100

Store            0.000000
Holiday_Flag     8.547009
Temperature      9.401709
Fuel_Price       9.401709
CPI              7.692308
Unemployment     0.000000
year            12.820513
month           12.820513
day             12.820513
day_of_week     12.820513
dtype: float64

In [17]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),  
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        
        ('OHE',OneHotEncoder(drop="first") )
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [18]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2)

In [19]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)


In [20]:
X_train[:1]

array([[ 1.58491115, -1.62687153,  0.8327713 ,  0.41468929, -1.18838799,
         0.60816202,  1.29483305,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  1.        ]])

In [21]:
X_test[:1]

array([[ 0.97240561,  0.71717657,  1.18677467, -0.72648205,  1.45619373,
        -0.4287761 , -0.79809403,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ]])

In [22]:
model=LinearRegression()
model.fit(X_train,Y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [23]:
print('Train Score :',model.score(X_train,Y_train))
print('Test Score :',model.score(X_test,Y_test))

Train Score : 0.9792723381396988
Test Score : 0.9207461733649066


In [24]:
features_names=numeric_features+list(preprocessor.named_transformers_['cat'].named_steps['OHE'].get_feature_names_out())


pd.DataFrame({'features':features_names,'Coefficient':model.coef_,'Intercept':model.intercept_}).sort_values(by='Coefficient',ascending=False)

,features,Coefficient,Intercept
10,Store_4.0,8.970690e+05,1.434206e+06
19,Store_14.0,7.264146e+05,1.434206e+06
18,Store_13.0,7.092792e+05,1.434206e+06
16,Store_10.0,6.841587e+05,1.434206e+06
25,Store_20.0,4.179266e+05,1.434206e+06
24,Store_19.0,2.474588e+05,1.434206e+06
8,Store_2.0,2.469645e+05,1.434206e+06
2,CPI,1.808313e+05,1.434206e+06
5,month,2.593009e+04,1.434206e+06
17,Store_11.0,2.345846e+04,1.434206e+06


In [25]:
# pd.DataFrame({'features':numeric_features+categorical_features,'Coefficient':model.coef_,'Intercept':model.intercept_}).sort_values(by='Coefficient',ascending=False)

In [26]:
model=Ridge()

params={'alpha':list(np.linspace(0,10,100))}
gridsearch=GridSearchCV(model,param_grid=params,cv=10)

In [27]:
gridsearch.fit(X_train,Y_train)
print('Train Score :',gridsearch.score(X_train,Y_train))
print('Test Score :',gridsearch.score(X_test,Y_test))
print('Best Params :',gridsearch.best_params_)
print('Best Score :',gridsearch.best_score_)
print('Best Estimator :',gridsearch.best_estimator_)


Train Score : 0.9779183305708207
Test Score : 0.925260342580321
Best Params : {'alpha': 0.0}
Best Score : 0.9360778428129967
Best Estimator : Ridge(alpha=0.0)


In [28]:
features_names=numeric_features+list(preprocessor.named_transformers_['cat'].named_steps['OHE'].get_feature_names_out())


pd.DataFrame({'features':features_names,'Coefficient':gridsearch.best_estimator_.coef_}).sort_values(by='Coefficient',ascending=False)

,features,Coefficient
7,day_of_week,1.452976e+20
10,Store_4.0,8.187881e+05
19,Store_14.0,7.149629e+05
18,Store_13.0,6.202943e+05
16,Store_10.0,6.019429e+05
25,Store_20.0,3.957438e+05
8,Store_2.0,2.781763e+05
24,Store_19.0,1.592939e+05
2,CPI,1.508755e+05
17,Store_11.0,3.295952e+04


# Selection des 3 meilleurs colonnes et remodelisation

In [36]:
X=data.loc[:,['Store','CPI','month']]
y=data['Weekly_Sales']

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [37]:
categorical_features=['Store']
numeric_features= ['CPI','month']

In [38]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),  
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        
        ('OHE',OneHotEncoder(drop="first") )
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [31]:
data.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,year,month,day,day_of_week
0,6.0,2011-02-18,1572117.54,NaN,59.61,3.045,214.777523,6.858,2011.0,2.0,18.0,4.0
1,13.0,2011-03-25,1807545.43,0.0,42.38,3.435,128.616064,7.470,2011.0,3.0,25.0,4.0
3,11.0,NaT,1244390.03,0.0,84.57,NaN,214.556497,7.346,NaN,NaN,NaN,NaN
4,6.0,2010-05-28,1644470.66,0.0,78.89,2.759,212.412888,7.092,2010.0,5.0,28.0,4.0
5,4.0,2010-05-28,1857533.70,0.0,NaN,2.756,126.160226,7.896,2010.0,5.0,28.0,4.0


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", Lasso(max_iter=10000, random_state=42)),
])


param_grid = {"model__alpha": np.linspace(0.0001, 10, 100)}
cv = KFold(n_splits=10, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train, Y_train)

print("Best params:", grid.best_params_)
print("CV best R² :", grid.best_score_)
print("Test R²    :", grid.score(X_test, Y_test))

Best params: {'model__alpha': 10.0}
CV best R² : 0.8927242492789975
Test R²    : 0.9547090679482971


In [ ]:
best_pipe = grid.best_estimator_
feature_names = best_pipe.named_steps["prep"].get_feature_names_out()
coefs = best_pipe.named_steps["model"].coef_

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
      .sort_values("coef", ascending=False)
)
coef_df.head(20)

,feature,coef
4,cat__Store_4.0,6.963024e+05
13,cat__Store_14.0,5.710539e+05
12,cat__Store_13.0,5.005896e+05
2,cat__Store_2.0,4.254710e+05
19,cat__Store_20.0,4.223420e+05
10,cat__Store_10.0,2.900643e+05
6,cat__Store_6.0,8.621417e+04
1,num__month,6.323308e+04
0,num__CPI,1.418130e+04
18,cat__Store_19.0,-6.462961e+04


# Conclusion 



``` Conclusion

L’analyse des ventes hebdomadaires de Walmart nous a permis de :

Préparer et nettoyer les données :

Conversion correcte des dates en variables dérivées (year, month, day, day_of_week).

Détection et retrait d’outliers (notamment sur Unemployment).

Gestion des valeurs manquantes via imputation et normalisation des variables numériques.

Encodage des variables catégorielles par OneHotEncoder.

Construire et comparer plusieurs modèles de régression :

Régression linéaire simple : scores corrects mais sensibles au surapprentissage.

Ridge Regression avec GridSearchCV : amélioration de la stabilité grâce à la régularisation ; optimisation du paramètre alpha a permis d’obtenir de meilleures performances globales.

Lasso Regression : mise en évidence des variables les plus influentes grâce à la sélection automatique de variables ; le pipeline complet (prétraitement + modèle) a permis d’éviter toute fuite de données et de fiabiliser la validation croisée.

Interprétation des coefficients :

Les variables liées au magasin (Store), au mois, et à l’indice des prix à la consommation (CPI) ressortent comme les plus déterminantes.

Le Lasso a permis de réduire l’importance de certaines variables peu explicatives, simplifiant ainsi le modèle.

```